In [1]:
import pandas as pd
import subprocess
from functools import reduce
import numpy as np
import umap
from plotnine import *
import os
import pickle

/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Explore 1000 genomes data

In [8]:
populations = pd.read_csv('/gpfs/commons/datasets/1000genomes/phase3/20131219.populations.tsv',sep='\t',engine='python')
superpopulations = pd.read_csv('/gpfs/commons/datasets/1000genomes/phase3/20131219.superpopulations.tsv', sep='\t',engine='python')

In [10]:
populations.groupby('Super Population')['Final Phase Samples'].sum()

Super Population
AFR    669.0
AMR    352.0
EAS    515.0
EUR    505.0
SAS    494.0
Name: Final Phase Samples, dtype: float64

Choose K=5 for admixture files; read in .ped and .map files

In [ ]:
def prep_1000genomes_bed_file(root_dir, output):
    # change map and ped files to bed format
    # major allele set to A2 (If a binary fileset was originally loaded, --keep-allele-order forces the original A1/A2 allele encoding to be preserved; otherwise, the major allele is set to A2)
    plink_extract = f'''
    module load plink/1.9 && plink --file {root_dir}/release-20130502-supporting/admixture_files/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05 \
        --make-bed \
        --out {output}
    '''
    result = subprocess.run(plink_extract, shell=True, check=True, executable="/bin/bash")


In [8]:
def read_in_igsr_samples(igsr_samples_filepath,bfile_path=None):
      # read in, subset to 2504, order correctly (if bfile path is not None)
      igsr_samples = pd.read_csv(igsr_samples_filepath,sep='\t')

      if bfile_path is not None:
            fam_df = pd.read_csv(f'{bfile_path}.fam',sep='\s+',header=None)
            fam_df.columns = ['FID','IID'] + fam_df.columns[2:].tolist()
            iid_order = fam_df['IID'].values
            igsr_samples = igsr_samples[igsr_samples['Sample name'].isin(iid_order)].copy()
            igsr_samples["Sample name"] = pd.Categorical(igsr_samples["Sample name"], categories=iid_order, ordered=True)
            igsr_samples = igsr_samples.sort_values("Sample name").reset_index(drop=True)
      igsr_samples["Superpopulation code"] = igsr_samples["Superpopulation code"].str.split(",").str[0] # chose first for sample with EUR,AFR superpopulation code
      igsr_samples.rename(columns={'Sample name':'IID'},inplace=True)
      igsr_samples['FID'] = igsr_samples['IID']

      return igsr_samples

def calculate_maf_by_superpop(igsr_samples_filepath,intermediate_file_dir,bfile_path,output):
      # Calculate allele frequencies in five superpopulations
      # generate superpopulation cluster file
      igsr_samples = read_in_igsr_samples(igsr_samples_filepath,bfile_path)
      igsr_samples[['FID','IID','Superpopulation code']].to_csv(f'{intermediate_file_dir}/superpop.clst',index=False,header=False,sep='\t')
      # calculate maf by superpop
      plink_freq = f''' module load plink/1.9 && 
      plink --bfile {bfile_path} \
            --freq \
            --within {intermediate_file_dir}/superpop.clst \
            --out {output}
      '''
      result = subprocess.run(plink_freq, shell=True, check=True, executable="/bin/bash")

      maf_by_superpop = pd.read_csv(f'{intermediate_file_dir}/maf_by_superpop.frq.strat',sep='\s+')
      return maf_by_superpop


In [ ]:
def sun_generate_sim_data(bfile_path, maf_by_superpop_filepath,
                          intermediate_file_dir,intermediate_file_suffix,output_dir,output_file_suffix,
                          ps,num_markers_assoc,e,extra_subgroups_size,M,num_clinical_assoc):
    '''
    Generate synthetic data similar to Sun et al. (Multi-view biclustering for genotype-phenotype association studies of complex diseases)
    using 1000 Genomes Phase 3 data. Use admixture files which contain 193634 markers with MAF>5% and 2504 individuals. 

    PARAMS:
    bfile_path: path to bfile for genetic data
    maf_by_superpop_filepath: plink generated .frq.strat file for maf within each superpopulation group. don't include .frq.strat suffix in filename.
    intermediate_file_dir: dir to write intermediate files to (when using plink for example)
    intermediate_file_suffix: such that if multiple simulations are created, each is distinctly defined
    intermediate_file_suffix: such that if multiple simulations are created, each is distinctly defined - suffic for C and simulated data pkl file
    output_dir: where to write output genetic data matrix (X) in form of plink bfile, and clinical data matrix C
    M: number of clinical features (right now assuming all from one domain & all binary)
    ps: variable controlling how much population stratification is affecting geno-pheno relationship (needs to be in range(0,1,size=0.1)) 
    (0-> pick SNPs in bottom 10% by allele frequency variance i.e. little pop. strat., 0.9-> pick SNPS in top 10% by allele frequency variance i.e. large pop. strat.)
    num_markers_assoc: number of markers with an associated with subtype classification (if rij>int(0.4*markers_assoc) then subject i in subgroup j)
    e: relative effect that genetic variation contributed to the effect of the phenotype. e in [0,1]. (decreased e means higher level of disagreement between genotypic and phenotypic subgroups)
    num_clinical_assoc: number of clinical features associated with subtype classification (same for all subtypes)
    extra_subgroups_size: number of people in s3 and s4 (selected at random)

    outputs:
    C: clinical data matrix (num samples x M)
    in pickle file (all simulation metadata):
    genetic_subgroups: genetic subgroup assignments for all individuals
    phenotypic_subgroups: phenotypic subgroup assignments for all individuals
    iid_order: IIDs in order (to align to clinical data matrix)
    markers_assoc_dict: for each genetic subgroup, IDs of genetic markers selected to be associated
    clinical_assoc_df: for each phenotypic subgroup, index of clinical variables selected to be associated (and their selected assoc. strength)

    '''
    assert num_clinical_assoc<M, "num_clinical_assoc cannot exceed M"

    # 1. Read in allele frequencies per 5 superpopulations to estimate af variance across groups
    maf_by_superpop = pd.read_csv(f'{maf_by_superpop_filepath}.frq.strat',sep='\s+')
    superpopulations = maf_by_superpop['CLST'].unique()
    maf_by_superpop = maf_by_superpop.pivot(index=['SNP'],columns='CLST',values='MAF').reset_index()
    assert maf_by_superpop.shape[0] == 193634
    maf_by_superpop['af_variance'] = maf_by_superpop[superpopulations].var(axis=1)
    maf_by_superpop['af_var_decile'] = (pd.qcut(maf_by_superpop['af_variance'], 10, labels=False))/10 # discretize into equal size buckets based on deciles


    # 2. Generate genetic subgroups
    genetic_subgroups = []
    markers_assoc_dict = {} # names of the markers that are associated with each subgroup
    for genetic_subgroup in range(2):
        # Select SNP group based on num_markers_assoc and ps
        assert maf_by_superpop[maf_by_superpop['af_var_decile']==ps].shape[0]>num_markers_assoc, f"number of genetic features assoc. ({num_markers_assoc}) is too large, only {maf_by_superpop[maf_by_superpop['af_var_decile']==ps].shape[0]} markers in decile {ps} group"
        markers_assoc = maf_by_superpop[maf_by_superpop['af_var_decile']==ps].sample(n=num_markers_assoc, replace=False)['SNP'].values.tolist()
        markers_assoc_dict[genetic_subgroup] = markers_assoc
        assert len(set(markers_assoc))==len(markers_assoc) # make sure ped file has unique rows

        
        # extract selected markers
        with open(f'{intermediate_file_dir}/markers_assoc_g{genetic_subgroup}_{intermediate_file_suffix}.txt','w') as f:
            for snp in markers_assoc:
                f.write(snp + "\n")
        
        # extract select markers and get marker values for each individual (0 - no copies of minor allele, 1 - 1 copy of minor allele, 2 - 2 copies of minor allele)
        plink_extract = f'''
        module load plink/1.9 && plink --bfile {bfile_path} \
            --extract {intermediate_file_dir}/markers_assoc_g{genetic_subgroup}_{intermediate_file_suffix}.txt \
            --make-bed \
            --recode A \
            --out {intermediate_file_dir}/subset_markers_g{genetic_subgroup}_{intermediate_file_suffix}
        '''
        result = subprocess.run(plink_extract, shell=True, check=True, executable="/bin/bash")

        # read in to get genotype counts
        raw = pd.read_csv(f'{intermediate_file_dir}/subset_markers_g{genetic_subgroup}_{intermediate_file_suffix}.raw',sep="\s+")
        geno = raw.drop(columns=['FID','IID','PAT','MAT','SEX','PHENOTYPE'])
        # assert they are all ≤ 0.5 (i.e., A1 is the minor allele)
        assert (geno.sum(axis=0) / (2 * geno.shape[0]) <= 0.5).all(), "Some SNPs have A1 frequency > 0.5"
        geno_cols = [col for col in raw.columns if not col in ['FID','IID','PAT','MAT','SEX','PHENOTYPE']]
        assert len(geno_cols) == num_markers_assoc
        raw[geno_cols] = (raw[geno_cols] > 0).astype(int) # recode s.t. values 1 and 2 map to 1
        genetic_subgroup_df = raw.set_index('IID')[geno_cols].sum(axis=1).reset_index(name=f'r')
        genetic_subgroup_df['genetic_subgroup'] = genetic_subgroup
        
        deciles, bins = pd.qcut(genetic_subgroup_df["r"], 10, labels=False, retbins=True)
        genetic_subgroup_df[f'subgroup'] =genetic_subgroup_df['r']>bins[-3] # Top 20% of people per r
        genetic_subgroups.append(genetic_subgroup_df)
    genetic_subgroups = pd.concat(genetic_subgroups)

    # 3. Generate phenotypic subgroups
    iid_order = raw.IID.values # can use raw file from whichever subgroup, since IID always in same order
    # can just use bins[-4] - somewhat equivalent to 7.5
    phenotypic_subgroups = []
    for phenotypic_subgroup in range(2):
        phenotypic_subgroup_df = genetic_subgroups[genetic_subgroups['genetic_subgroup']==phenotypic_subgroup][['IID','r']].copy()
        phenotypic_subgroup_df['phenotypic_subgroup'] = phenotypic_subgroup
        phenotypic_subgroup_df['subgroup'] = phenotypic_subgroup_df['r']*e + np.random.randn(len(phenotypic_subgroup_df)) > bins[-4]*e
        phenotypic_subgroups.append(phenotypic_subgroup_df)
    for phenotypic_subgroup in range(2,4):
        # randomly select extra_subgroups_size people
        randomly_selected = pd.Series(iid_order).sample(extra_subgroups_size).values.tolist() 
        phenotypic_subgroup_df = pd.DataFrame(iid_order,columns=['IID'])
        phenotypic_subgroup_df['r'] = None
        phenotypic_subgroup_df['phenotypic_subgroup'] = phenotypic_subgroup
        phenotypic_subgroup_df['subgroup'] = phenotypic_subgroup_df['IID'].isin(randomly_selected)
        phenotypic_subgroups.append(phenotypic_subgroup_df)
    phenotypic_subgroups = pd.concat(phenotypic_subgroups)

    # 4. simulate M binary clinical features
    # start with baseline probabiliyies
    probs = np.full((len(iid_order), M), 0.1, dtype=float) # baseline prob of clinical feature is 0.1
    clinical_assoc_df_rows = [] # index of clinical vars that are associated (and their strength)
    for phenotypic_subgroup in range(4):
        # index of randomly chosen, associated clinical variables 
        assoc_idx = np.random.choice(M, size=num_clinical_assoc, replace=False) 
        np.random.shuffle(assoc_idx)       
        n1 = num_clinical_assoc // 3 # 1/3 who get P 0.6
        n2 = 2 * num_clinical_assoc // 3 # 1/3 who get P 0.5
        # get those who are in the subgroup
        mask = ((phenotypic_subgroups["phenotypic_subgroup"] == phenotypic_subgroup) & (phenotypic_subgroups["subgroup"]))
        subj_ids = phenotypic_subgroups.loc[mask, "IID"].unique()
        # map subject IDs to row indices 
        row_idx = [i for i, iid in enumerate(iid_order) if iid in subj_ids]
        probs[np.ix_(row_idx, assoc_idx[:n1])] = 0.6
        probs[np.ix_(row_idx, assoc_idx[n1:n2])] = 0.5
        probs[np.ix_(row_idx, assoc_idx[n2:])] = 0.4
        clinical_assoc_df_rows += [
        {"phenotypic_subgroup": phenotypic_subgroup, "strength": 0.6, "indices": assoc_idx[:n1]},
        {"phenotypic_subgroup": phenotypic_subgroup, "strength": 0.5, "indices": assoc_idx[n1:n2]},
        {"phenotypic_subgroup": phenotypic_subgroup, "strength": 0.4, "indices": assoc_idx[n2:]},
        ]
    clinical_assoc_df = pd.DataFrame(clinical_assoc_df_rows)

    # write C
    C = (np.random.rand(len(iid_order), M) < probs).astype(int)
    np.save(f"{output_dir}/C_{output_file_suffix}.npy", C)

    # write simulation metadata
    bundle = {
    "iid_order": iid_order,                          # list/array
    "genetic_subgroups": genetic_subgroups,          # list/array/Series
    "phenotypic_subgroups": phenotypic_subgroups,    # list/array/Series
    "markers_assoc": markers_assoc_dict,             # dict: gen_subgroup -> [marker_id, ...]
    "clinical_assoc": clinical_assoc_df              # pandas DataFrame
    }
    with open(f"{output_dir}/simulation_metadata_{output_file_suffix}.pkl", "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

    return genetic_subgroups, phenotypic_subgroups, C, iid_order, markers_assoc_dict, clinical_assoc_df 
    

In [ ]:
intermediate_file_dir ='/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/intermediate_plink'
root_dir = '/gpfs/commons/datasets/1000genomes'
igsr_samples_filepath = '/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/input/igsr_samples.tsv'
output_dir = '/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output'

genetic_subgroups,phenotypic_subgroups,C,iid_order,markers_assoc_dict, clinical_assoc_df = sun_generate_sim_data(bfile_path=f'{output_dir}/X', maf_by_superpop_filepath=f'{intermediate_file_dir}/maf_by_superpop', intermediate_file_dir=intermediate_file_dir,
                                        intermediate_file_suffix='',output_dir = output_dir,  output_file_suffix='TEST',
                                      ps=0, num_markers_assoc=2000, e=0.5,extra_subgroups_size=200, M=100,num_clinical_assoc=10) # intermediate_file_suffix important if run in parallel


## Determine the effect of ps on confounding by superpopulation

In [135]:
ps_list = np.round(np.arange(0, 1.0, 0.1),1)
e_list = np.round(np.arange(0, 1.1, 0.1),1)

In [9]:
igsr_samples = read_in_igsr_samples(igsr_samples_filepath, bfile_path=f'{output_dir}/X')

In [177]:
def generate_umap_plot(mode, var_list, color_col, color_label, output_dir):
    '''Generate UMAP plot of C across different values of variable spcified in 'mode' 
    (values in var_list), colored by color_df (which must have a column IID)'''
    plot_dfs = []
    for var in var_list:
        if mode == 'ps':
            output_suffix = f'ps_{var}_e_0.5'
            with open(f"{output_dir}/simulation_metadata_{output_suffix}.pkl", "rb") as f:
                simulation_metadata = pickle.load(f)
            color_df = read_in_igsr_samples(igsr_samples_filepath, bfile_path=f'{output_dir}/X')
        else:
            assert mode=='e', "only works with modes ps and e so far"
            output_suffix = f'ps_0.5_e_{var}'
            with open(f"{output_dir}/simulation_metadata_{output_suffix}.pkl", "rb") as f:
                simulation_metadata = pickle.load(f)
            genetic_subgroups = simulation_metadata['genetic_subgroups']
            genetic_subgroups['subgroup_value'] = np.where(genetic_subgroups['subgroup'],genetic_subgroups['genetic_subgroup'],'') # change this to one label per person
            genetic_subgroups_concat = genetic_subgroups.groupby('IID')['subgroup_value'].apply(clean_join).reset_index()
            genetic_subgroups_concat["IID"] = pd.Categorical(genetic_subgroups_concat["IID"], categories=iid_order, ordered=True)
            color_df = genetic_subgroups_concat.copy()
        C = np.load(f'{output_dir}/C_{output_suffix}.npy')# pick e=0.5

        reducer = umap.UMAP()
        embedding = reducer.fit_transform(C)

        plot_df = (pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"], index=simulation_metadata['iid_order']).
                    rename_axis("IID").reset_index().merge(color_df[['IID',color_col]], on='IID',how='inner'))
        
        assert plot_df[plot_df[color_col].isna()].shape[0] == 0
        plot_df[mode] = var
        plot_dfs.append(plot_df)
    plot_dfs = pd.concat(plot_dfs)

    # --- Plot with plotnine ---
    p = (
        ggplot(plot_dfs, aes("UMAP1", "UMAP2", color=color_col))
        + geom_point(alpha=0.7, size=2)
        + labs(title=r"UMAP of clinical data at different $p_s$ levels", color=color_label)
        + facet_wrap(f'~{mode}',ncol=2,scales='free')
        + theme_minimal()
        + theme(figure_size=(8, 12),legend_title=element_text(size=9))
    )
    p.save(f'{output_dir}/umap_clinical_{mode}.pdf',dpi=300)


In [ ]:
generate_umap_plot(mode='ps', var_list=[0.1,0.5], color_col='Superpopulation name', color_label='Superpopulation', output_dir=output_dir)

In [178]:
generate_umap_plot(mode='e', var_list=[0.1,0.5], color_col='subgroup_value', color_label='Genetic Subgroup', output_dir=output_dir)

/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 8 x 12 in image.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/umap_clinical_e.pdf


In [137]:
def clean_join(values):
    # remove empty strings
    vals = sorted(set(v for v in values if v != ''), key=lambda x: int(x))
    return ','.join(vals)

### Run GWAS with and without PCs and see how much genetic coefficients change 
At different levels of population stratification

In [ ]:
pcs = pd.read_csv(f'{output_dir}/pcs.txt',sep='\t')
covar = pcs.merge(igsr_samples[['IID','Sex']], on='IID',how='inner')
# probably cleaner way to do this
covar[['FID','IID']+[f'PC{i}' for i in range(1,11)]+['Sex']].set_index('FID').to_csv(f'{intermediate_file_dir}/COVARIATE_FILE')
covar[['FID','IID']+['Sex']].set_index('FID').to_csv(f'{intermediate_file_dir}/COVARIATE_FILE_NOPS')

for ps, e in product(ps_list, e_list):
    output_suffix = f'ps_{ps}_e_{e}'
    for cov_included in [False,True]:
        if cov_included:
            cov_file_suffix = ''
        else:
            cov_file_suffix = '_NOPS'
        for phenotypic_subgroup in range(4): # subgroup 0 & 1 should be affected, subgroup 2 & 3 shouldn't be affected (randomly chosen people)
            with open(f"{output_dir}/simulation_metadata_{output_suffix}.pkl", "rb") as f:
                simulation_metadata = pickle.load(f)
            phenotypic_subgroups = simulation_metadata['phenotypic_subgroups']
            pheno = phenotypic_subgroups[phenotypic_subgroups['phenotypic_subgroup']==phenotypic_subgroup][['IID','subgroup']].rename(columns={'subgroup':'Phenotype'})
            pheno['FID'] = pheno['IID']
            pheno['Phenotype'] = pheno['Phenotype'].astype(int)
            pheno[['FID','IID','Phenotype']].set_index('FID').to_csv(f'{intermediate_file_dir}/PHENOTYPE_FILE_Subgroup{phenotypic_subgroup}')

            result = subprocess.run(f'module unload plink && module load plink/2.0a5.13 && plink --bfile {output_dir}/X\
                                --covar {intermediate_file_dir}/COVARIATE_FILE{cov_file_suffix} --covar-variance-standardize\
                                --pheno {intermediate_file_dir}/PHENOTYPE_FILE_Subgroup{phenotypic_subgroup}\
                                --glm omit-ref\
                                --out {output_dir}/GWAS_RESULTS/PhenotypicSubgroup_{phenotypic_subgroup}_Geno_Cov_{cov_included}_ps_{ps}_e_{e}\
                                --1 --no-pheno', shell=True, capture_output=True, text=True, executable='/bin/bash')
            result.check_returncode()


for subgroup 0, what's the difference in coeff effect size when covs included or not

In [79]:
# general idea: fit crude and adjusted model and if beta changes by more than 10%, we conclude that ancestry is a confounder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

phenotypic_subgroup = 0
all_plink_results = []
for cov_included in [False,True]:
    plink_results = pd.read_csv(f'{os.path.dirname(intermediate_file_dir)}/output/RESULTS_FILE_Subgroup{phenotypic_subgroup}_Geno_Cov{cov_included}.Phenotype.glm.logistic.hybrid',sep='\t')
    print(plink_results.shape)
    plink_results = plink_results[plink_results['TEST']=='ADD'].copy()
    plink_results['marker_assoc'] = plink_results['ID'].isin(markers_assoc_dict[phenotypic_subgroup]) # ground truth from simulations
    # using significance level 5e-8, what is the accuracy of association test
    plink_results['significant'] = plink_results['P'] < 5e-8 # predicted associated
    print(plink_results[plink_results['P'] < 5e-8].shape[0])
    accuracy = accuracy_score(plink_results["marker_assoc"], plink_results["significant"])
    precision = precision_score(plink_results["marker_assoc"], plink_results["significant"])
    recall = recall_score(plink_results["marker_assoc"], plink_results["significant"])
    f1 = f1_score(plink_results["marker_assoc"], plink_results["significant"])
    print(cov_included)
    print(accuracy)
    print(precision)
    print(recall)
    print(f1)
    plink_results['cov_included'] = cov_included
    all_plink_results.append(plink_results)
all_plink_results = pd.concat(all_plink_results)

(387268, 17)
128086
False
0.3440924631004886
0.012023171931358618
0.77
0.02367664468121089
(1161804, 17)
0
True
0.989671235423531
0.0
0.0
0.0


/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.


In [83]:
wide = (all_plink_results
        .pivot(index='ID', columns='cov_included', values='OR')
        .rename(columns={False:'OR_no_cov', True:'OR_with_cov'})
        .reset_index()
)

# Calculate relative change
wide['relative_change'] = abs(wide['OR_with_cov'] - wide['OR_no_cov']) / wide['OR_no_cov']
print(wide['relative_change'].mean())
wide

0.7869915795402833


cov_included,ID,OR_no_cov,OR_with_cov,relative_change
0,10:10002676,0.407327,0.793863,0.948957
1,10:100033081,60.261000,0.607603,0.989917
2,10:10004698,68.774400,0.331431,0.995181
3,10:100068607,0.754749,1.165830,0.544659
4,10:100076982,0.508965,0.817929,0.607044
...,...,...,...,...
193629,9:99959341,0.811283,1.173930,0.447004
193630,9:99979829,124.290000,0.521338,0.995805
193631,9:99981843,0.515383,1.016670,0.972649
193632,9:9998777,0.437162,0.953639,1.181432


In [85]:
# should see no change wi†h this subgroup
phenotypic_subgroup = 2
all_plink_results = []
for cov_included in [False,True]:
    plink_results = pd.read_csv(f'{os.path.dirname(intermediate_file_dir)}/output/RESULTS_FILE_Subgroup{phenotypic_subgroup}_Geno_Cov{cov_included}.Phenotype.glm.logistic.hybrid',sep='\t')
    print(plink_results.shape)
    plink_results = plink_results[plink_results['TEST']=='ADD'].copy()
    plink_results['marker_assoc'] = False # ground truth from simulations
    # using significance level 5e-8, what is the accuracy of association test
    plink_results['significant'] = plink_results['P'] < 5e-8 # predicted associated
    print(plink_results[plink_results['P'] < 5e-8].shape[0])
    accuracy = accuracy_score(plink_results["marker_assoc"], plink_results["significant"])
    precision = precision_score(plink_results["marker_assoc"], plink_results["significant"])
    recall = recall_score(plink_results["marker_assoc"], plink_results["significant"])
    f1 = f1_score(plink_results["marker_assoc"], plink_results["significant"])
    print(cov_included)
    print(accuracy)
    print(precision)
    print(recall)
    print(f1)
    plink_results['cov_included'] = cov_included
    all_plink_results.append(plink_results)
all_plink_results = pd.concat(all_plink_results)

(387268, 17)
0


/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.


False
1.0
0.0
0.0
0.0
(1161804, 17)
0
True
1.0
0.0
0.0
0.0


/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.


In [86]:
wide = (all_plink_results
        .pivot(index='ID', columns='cov_included', values='OR')
        .rename(columns={False:'OR_no_cov', True:'OR_with_cov'})
        .reset_index()
)

# Calculate relative change
wide['relative_change'] = abs(wide['OR_with_cov'] - wide['OR_no_cov']) / wide['OR_no_cov']
print(wide['relative_change'].mean())
wide

0.029815247828271215


cov_included,ID,OR_no_cov,OR_with_cov,relative_change
0,10:10002676,1.142180,1.118660,0.020592
1,10:100033081,0.849409,0.803559,0.053979
2,10:10004698,0.686901,0.575850,0.161670
3,10:100068607,0.941181,0.875650,0.069626
4,10:100076982,0.994909,1.020650,0.025873
...,...,...,...,...
193629,9:99959341,1.286830,1.423020,0.105834
193630,9:99979829,0.730324,0.659314,0.097231
193631,9:99981843,1.011900,1.014090,0.002164
193632,9:9998777,0.975574,0.920400,0.056555


In [ ]:
# analyze with effect of ps


In [ ]:
# also check that the correct markers assoc and conditions assoc. are showing up

### Run GWAS at different levels of e and see how strength of association changes

In [93]:
import pickle
bundle = {
    "iid_order": iid_order,                          # list/array
    "genetic_subgroups": genetic_subgroups,          # list/array/Series
    "phenotypic_subgroups": phenotypic_subgroups,    # list/array/Series
    "markers_assoc": markers_assoc_dict,             # dict: gen_subgroup -> [marker_id, ...]
    "clinical_assoc": clinical_assoc_df              # pandas DataFrame
}

with open("/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/bundle.pkl", "wb") as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

# load
with open("/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/bundle.pkl", "rb") as f:
    data = pickle.load(f)
data

{'iid_order': array(['HG02922', 'HG02923', 'HG02938', ..., 'HG04210', 'HG04227',
        'HG04229'], dtype=object),
 'genetic_subgroups':           IID    r  genetic_subgroup  subgroup subgroup_value
 0     HG02922  844                 0      True              0
 1     HG02923  837                 0      True              0
 2     HG02938  867                 0      True              0
 3     HG02941  839                 0      True              0
 4     HG02943  850                 0      True              0
 ...       ...  ...               ...       ...            ...
 2499  HG04106  683                 1     False               
 2500  HG04107  722                 1     False               
 2501  HG04210  690                 1     False               
 2502  HG04227  709                 1     False               
 2503  HG04229  730                 1     False               
 
 [5008 rows x 5 columns],
 'phenotypic_subgroups':           IID     r  phenotypic_subgroup  subgroup
 0 

### Algorithm creation

In [ ]:
# empirically, show monotone decrease of loss and stabilization over iterations
# show stability of parameter norms (U^t-U^(t-1))

# JointMF
from algorithms.JointMF import *
writer = None

# read in simulated data
ps = 0.7
e = 0.5
C = np.load(f'{output_dir}/C_ps_{ps}_e_{e}.npy')
with open(f"{output_dir}/simulation_metadata_ps_{ps}_e_{e}.pkl", "rb") as f:
    simulation_metadata = pickle.load(f)

# read in covar file and X
covar = pd.read_csv(f'{intermediate_file_dir}/COVARIATE_FILE')
assert all(covar['IID'].values == simulation_metadata['iid_order']) # make sure all in same order
covar["Sex"] = (covar["Sex"] == "female").astype(int)
Z = covar.drop(['FID','IID'],axis=1).to_numpy()

plink_extract = f'''
module load plink/1.9 && plink --bfile {output_dir}/X \
    --recode A \
    --out {output_dir}/X
'''
result = subprocess.run(plink_extract, shell=True, check=True, executable="/bin/bash")
A,B,D,B_prime, D_prime, loss_history, success, message, max_mem, cpu_time, user_time = jmf(X, C, Z, rank=4, method='L-BFGS-B', writer=writer, options={'maxcor':5, 'maxiter':1000, 'gtol':1e-5, 'maxls':10, 'ftol':1e-5})


PLINK v1.90b6.21 64-bit (19 Oct 2020)          www.cog-genomics.org/plink/1.9/
(C) 2005-2020 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/X.log.
Options in effect:
  --bfile /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/X
  --out /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/X
  --recode A

385553 MB RAM detected; reserving 192776 MB for main workspace.
193634 variants loaded from .bim file.
2504 people (0 males, 0 females, 2504 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/X.nosex
.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2504 founders and 0 nonfounders present.
Calculating allele frequencies... 1011121314151617181920212223242526272829303132333435363738394041

In [16]:
X_df = pd.read_csv(f'{output_dir}/X.raw', sep='\s+')
X_df


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [24]:
igsr_samples = read_in_igsr_samples(igsr_samples_filepath, bfile_path=f'{output_dir}/X')
igsr_samples['Superpopulation code'].unique()

array(['AFR', 'AMR', 'EAS', 'EUR', 'SAS'], dtype=object)

In [ ]:
ps_list = np.round(np.arange(0, 1.0, 0.1),1)
e_list = np.round(np.arange(0, 1.1, 0.1),1)
from simulations.genomes1000_sim import *
generate_umap_plot(mode='ps', var_list=ps_list, color_col='Superpopulation code',
                        color_label='Superpopulation', output_dir=output_dir,igsr_samples_filepath=igsr_samples_filepath)

# For e
generate_umap_plot(mode='e', var_list=e_list, color_col='subgroup_value', 
                        color_label='Genetic Subgroup', output_dir=output_dir)

/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 8 x 12 in image.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/umap_clinical_ps.pdf
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 8 x 12 in image.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations/output/umap_clinical_e.pdf
